In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/gazizovvyacheslav/dataset3/sessions_with_session_and_install_features.parquet
/kaggle/input/datasets/gazizovvyacheslav/dataset3/sessions_with_session_and_install_features.csv
/kaggle/input/datasets/kravchukilya/events/events.csv
/kaggle/input/datasets/kravchukilya/events-2/events-2.csv


Сразу придумаем признаки перед чтением, тк будем читать с помощью duckdb

most_common_event_name - название самого частого события

most_common_connection_type - самый частый тип подключения
<!-- 
first_event_name - первое событие в сессии

last_event_name - последнее событие в сессии -->

most_common_event_count - сколько раз встретилось самое частое событие

most_common_event_share - доля самого частого события среди всех событий сессии

events_total - общее число событий в сессии

unique_events_count - число разных типов событий

events_span_sec - время между первым и последним событием

first_event_timestamp - timestamp первого события

last_event_timestamp - timestamp последнего события

unique_device_models - число разных моделей устройства

unique_device_types - число разных типов устройства

unique_app_versions - число разных версий приложения

unique_countries - число разных стран

unique_cities - число разных городов

unique_connection_types - число разных типов соединения


DuckDB будем юзать потому что он позволяет агрегировать большие события без загрузки всего файла как например в pandas

In [2]:
import os
# import time
import pandas as pd
import duckdb

con = duckdb.connect()

con.execute("SET preserve_insertion_order=false;")
con.execute("SET threads=2;")
con.execute("SET memory_limit='12GB';")
con.execute("PRAGMA enable_progress_bar;")

events1_path = "/kaggle/input/datasets/kravchukilya/events/events.csv"
events2_path = "/kaggle/input/datasets/kravchukilya/events-2/events-2.csv"

Функция строит базовые признаки по каждой сессии:

количество событий, число уникальных событий, время начала/конца и разнообразие устройств/географии и тд

In [3]:
def build_events_base(events_path, output_path):
    q = f"""
    CREATE OR REPLACE TEMP TABLE tmp AS
    SELECT *
    FROM read_csv_auto('{events_path}');
    
    CREATE OR REPLACE TEMP TABLE tmp1 AS
    SELECT
        CAST(installation_id AS VARCHAR) AS installation_id,
        CAST(session_id AS VARCHAR) AS session_id,
        COUNT(*) AS events_total,
        COUNT(DISTINCT event_name) AS unique_events_count,
        MIN(CAST(event_datetime AS TIMESTAMP)) AS first_event_time,
        MAX(CAST(event_datetime AS TIMESTAMP)) AS last_event_time,
        MIN(CAST(event_timestamp AS BIGINT)) AS first_event_timestamp,
        MAX(CAST(event_timestamp AS BIGINT)) AS last_event_timestamp,
        COUNT(DISTINCT device_model) AS unique_device_models,
        COUNT(DISTINCT device_type) AS unique_device_types,
        COUNT(DISTINCT app_version_name) AS unique_app_versions,
        COUNT(DISTINCT country_iso_code) AS unique_countries,
        COUNT(DISTINCT city) AS unique_cities,
        COUNT(DISTINCT connection_type) AS unique_connection_types
    FROM tmp
    GROUP BY installation_id, session_id;
    COPY tmp1 TO '{output_path}' (FORMAT PARQUET);
    """

    con.execute(q)
    print("путь :", output_path)

Здесь находим признак - самое частое событие внутри каждой сессии

In [4]:
def build_most_common_event(events_path, output_path):
    q = f"""
    CREATE OR REPLACE TEMP TABLE tmp AS
    SELECT CAST(installation_id AS VARCHAR) AS installation_id,
        CAST(session_id AS VARCHAR) AS session_id,
        event_name,
        COUNT(*) AS event_count
    FROM read_csv_auto('{events_path}')
    GROUP BY installation_id, session_id, event_name;
    
    CREATE OR REPLACE TEMP TABLE tmp1 AS
    SELECT installation_id, session_id,
        event_name AS most_common_event_name,
        event_count AS most_common_event_count
    FROM (SELECT *,
        ROW_NUMBER() OVER (
        PARTITION BY installation_id, session_id
        ORDER BY event_count DESC, event_name ASC
        ) AS rn
        FROM tmp)
    WHERE rn = 1;
    
    COPY tmp1 TO '{output_path}' (FORMAT PARQUET);
    """

    con.execute(q)
    print("путь:", output_path)

А здесб -  самый частый тип подключения в сессии

In [5]:
def build_most_common_connection(events_path, output_path):
    q = f"""
    CREATE OR REPLACE TEMP TABLE tmp AS
    SELECT CAST(installation_id AS VARCHAR) AS installation_id,
        CAST(session_id AS VARCHAR) AS session_id,
        connection_type,
        COUNT(*) AS connection_count
    FROM read_csv_auto('{events_path}', all_varchar=true)
    GROUP BY installation_id, session_id, connection_type;
    
    CREATE OR REPLACE TEMP TABLE tmp1 AS
    SELECT installation_id, session_id,
        connection_type AS most_common_connection_type
    FROM (SELECT *,
        ROW_NUMBER() OVER (
        PARTITION BY installation_id, session_id
        ORDER BY connection_count DESC, connection_type ASC
        ) AS rn
        FROM tmp)
    WHERE rn = 1;
    COPY tmp1 TO '{output_path}' (FORMAT PARQUET);
    """

    con.execute(q)
    print("путь:", output_path)


In [ ]:
build_events_base(events1_path, "/kaggle/working/events1_base.parquet")

build_most_common_event(events1_path, "/kaggle/working/events1_most_common_event.parquet")

build_most_common_connection(events1_path, "/kaggle/working/events1_most_common_connection.parquet")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Загружаем промежуточные таблицы с признаками для первого файла событий и объединяем их в одну таблицу признаков (сессии)

In [ ]:
events1_base = pd.read_parquet("/kaggle/working/events1_base.parquet")
events1_mce = pd.read_parquet("/kaggle/working/events1_most_common_event.parquet")
events1_mcc = pd.read_parquet("/kaggle/working/events1_most_common_connection.parquet")

events1_features = (
    events1_base
    .merge(events1_mce, on=["installation_id", "session_id"], how="left")
    .merge(events1_mcc, on=["installation_id", "session_id"], how="left")
)

events1_features["events_span_sec"] = (events1_features["last_event_time"] - events1_features["first_event_time"]).dt.total_seconds()

events1_features["most_common_event_share"] = (events1_features["most_common_event_count"] / events1_features["events_total"])

events1_features = events1_features[
    [
        "installation_id",
        "session_id",
        "most_common_event_name",
        "most_common_connection_type",
        "most_common_event_count",
        "most_common_event_share",
        "events_total",
        "unique_events_count",
        "events_span_sec",
        "first_event_timestamp",
        "last_event_timestamp",
        "unique_device_models",
        "unique_device_types",
        "unique_app_versions",
        "unique_countries",
        "unique_cities",
        "unique_connection_types",
    ]
]

events1_features.to_parquet("/kaggle/working/events1_features_final.parquet", index=False)


Теперь для второй таблички

In [ ]:
build_events_base(events2_path, "/kaggle/working/events2_base.parquet")

build_most_common_event(events2_path, "/kaggle/working/events2_most_common_event.parquet")

build_most_common_connection(events2_path,"/kaggle/working/events2_most_common_connection.parquet")


In [ ]:
events2_base = pd.read_parquet("/kaggle/working/events2_base.parquet")
events2_mce = pd.read_parquet("/kaggle/working/events2_most_common_event.parquet")
events2_mcc = pd.read_parquet("/kaggle/working/events2_most_common_connection.parquet")

events2_features = (events2_base.merge(events2_mce, on=["installation_id", "session_id"], how="left").merge(events2_mcc, on=["installation_id", "session_id"], how="left"))
events2_features["events_span_sec"] = (events2_features["last_event_time"] - events2_features["first_event_time"]).dt.total_seconds()
events2_features["most_common_event_share"] = (events2_features["most_common_event_count"] / events2_features["events_total"])

events2_features = events2_features[
    [
        "installation_id",
        "session_id",
        "most_common_event_name",
        "most_common_connection_type",
        "most_common_event_count",
        "most_common_event_share",
        "events_total",
        "unique_events_count",
        "events_span_sec",
        "first_event_timestamp",
        "last_event_timestamp",
        "unique_device_models",
        "unique_device_types",
        "unique_app_versions",
        "unique_countries",
        "unique_cities",
        "unique_connection_types",
    ]
]

events2_features.to_parquet("/kaggle/working/events2_features_final.parquet", index=False)

Теперь делаем общий файл

In [ ]:
sessions = pd.read_parquet("/kaggle/input/datasets/gazizovvyacheslav/dataset3/sessions_with_session_and_install_features.parquet")

events1 = pd.read_parquet("/kaggle/working/events1_features_final.parquet")
events2 = pd.read_parquet("/kaggle/working/events2_features_final.parquet")

events_features = pd.concat([events1, events2], ignore_index=True)

events_features = events_features.groupby(["installation_id", "session_id"]).agg({
    "most_common_event_name": "first",
    "most_common_connection_type": "first",
    "most_common_event_count": "max",
    "most_common_event_share": "max",
    "events_total": "sum",
    "unique_events_count": "max",
    "events_span_sec": "max",
    "first_event_timestamp": "min",
    "last_event_timestamp": "max",
    "unique_device_models": "max",
    "unique_device_types": "max",
    "unique_app_versions": "max",
    "unique_countries": "max",
    "unique_cities": "max",
    "unique_connection_types": "max"
}).reset_index()

sessions["installation_id"] = sessions["installation_id"].astype(str)
sessions["session_id"] = sessions["session_id"].astype(str)

events_features["installation_id"] = events_features["installation_id"].astype(str)
events_features["session_id"] = events_features["session_id"].astype(str)

sessions = sessions.merge(events_features, on=["installation_id", "session_id"], how="left")

In [ ]:
cols = [
    "most_common_event_name",
    "most_common_connection_type",
    "most_common_event_count",
    "most_common_event_share",
    "events_total",
    "unique_events_count",
    "events_span_sec",
    "first_event_timestamp",
    "last_event_timestamp",
    "unique_device_models",
    "unique_device_types",
    "unique_app_versions",
    "unique_countries",
    "unique_cities",
    "unique_connection_types"
]

sessions[cols].isna().mean().sort_values(ascending=False)

In [ ]:
sessions.to_parquet(
    "/kaggle/working/sessions_with_all_features.parquet",
    index=False
)